# run02のattention_maskエラーを修正して再開
元のrun02と同じColab T4サーバーに接続して、下のセルを実行してください。
ライブラリ再インストール・Stage 1復元・ZIPアップロードは不要。学習条件は変更しません。
元のnotebookとの同時実行はしないでください。

In [ ]:
from pathlib import Path
import sys, json
worker = Path('/content/run02_worker.py')
if not worker.is_file():
    raise RuntimeError('元のrun02と同じColabランタイムに接続してください。')
OLD = "        mask = batch['attention_mask'].bool()\n        if not torch.equal(batch['input_ids'][mask], batch['labels'][mask]):\n            raise RuntimeError('想定外のlossマスクを検出しました。')\n        if not torch.all(batch['labels'][~mask] == -100):\n            raise RuntimeError('paddingのlossマスクが不正です。')\n        n = int(mask.sum())\n        if (batch['input_ids'][mask].tolist() != tokenized[i]['input_ids']\n                or batch['input_ids'][0,n-1].item() != tokenizer.eos_token_id):\n            raise RuntimeError('データの切詰め/EOS欠落を検出しました。')"
NEW = "        # A one-example padding-free batch can omit attention_mask.\n        ids, labels = batch['input_ids'], batch['labels']\n        if ids.ndim != 2 or ids.shape[0] != 1 or labels.shape != ids.shape:\n            raise RuntimeError('検査用の単一example batchの形状が不正です。')\n        if 'attention_mask' in batch:\n            mask = batch['attention_mask'].bool()\n        else:\n            # Do not infer padding from token IDs: pad and EOS can share an ID.\n            if ids.numel() != len(tokenized[i]['input_ids']):\n                raise RuntimeError('attention_maskなしで系列長が変わっています。')\n            mask = torch.ones_like(ids, dtype=torch.bool)\n        actual_ids, actual_labels = ids[mask], labels[mask]\n        if (actual_ids.tolist() != tokenized[i]['input_ids']\n                or actual_ids[-1].item() != tokenizer.eos_token_id):\n            raise RuntimeError('データの切詰め/EOS欠落を検出しました。')\n        expected_labels = actual_ids.clone()\n        # The first label has no preceding token and is never a causal target.\n        # Padding-free collators may explicitly ignore that boundary label.\n        if actual_labels[0].item() == -100:\n            expected_labels[0] = -100\n        if not torch.equal(expected_labels, actual_labels):\n            raise RuntimeError('想定外のlossマスクを検出しました。')\n        if not torch.all(labels[~mask] == -100):\n            raise RuntimeError('paddingのlossマスクが不正です。')"
source = worker.read_text()
if OLD in source:
    source = source.replace(OLD, NEW)
    compile(source, str(worker), 'exec')
    worker.write_text(source)
elif NEW not in source:
    raise RuntimeError('想定と異なるworkerです。書き換えを中止しました。')
print('学習前のbatch検査を修正しました。')
"""Colab launcher utilities; safe bundle restoration and isolated child processes."""
import hashlib
import json
import os
import signal
import subprocess
import sys
from pathlib import Path, PurePosixPath
from zipfile import ZipFile


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(8*1024*1024), b''):
            h.update(b)
    return h.hexdigest()


def restore_bundle(bundle, workspace, expected_sha256):
    """Exact trusted archive only. Verify every member before writing; never overwrite differences."""
    if sha256_file(bundle) != expected_sha256:
        raise RuntimeError('ZIPの内容が用意したものと異なります。run02_colab_inputs.zipを確認してください。')
    workspace = Path(workspace).resolve()
    with ZipFile(bundle) as z:
        manifest = json.loads(z.read('bundle_manifest.json'))
        if sorted(z.namelist()) != sorted(['bundle_manifest.json', *manifest['files']]):
            raise RuntimeError('ZIP内のファイル一覧がmanifestと一致しません。')
        pending = []
        for name, checksum in manifest['files'].items():
            relative = PurePosixPath(name)
            if relative.is_absolute() or '..' in relative.parts:
                raise RuntimeError('不正なZIP内パスです。')
            destination = workspace.joinpath(*relative.parts)
            if not destination.resolve().is_relative_to(workspace):
                raise RuntimeError('保存先がworkspace外です。')
            payload = z.read(name)
            if hashlib.sha256(payload).hexdigest() != checksum:
                raise RuntimeError('ZIP内ファイルのハッシュが一致しません。')
            if destination.exists():
                if sha256_file(destination) != checksum:
                    raise RuntimeError('Driveに別内容のファイルがあります。上書きしません: '+name)
            else:
                pending.append(name)
        # All conflicts checked before any writes.
        for name in pending:
            target = workspace/name
            target.parent.mkdir(parents=True, exist_ok=True)
            tmp = target.with_name(target.name+'.run02_tmp')
            tmp.write_bytes(z.read(name))
            os.replace(tmp, target)
        return {'added': len(pending), 'already_present': len(manifest['files'])-len(pending)}


def run_logged(args, log_path, echo=True):
    """Fresh interpreter means no manual kernel restart; SIGINT terminates its whole process group."""
    path = Path(log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', buffering=1) as log:
        log.write('\n=== process start ===\n')
        process = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, start_new_session=True)
        try:
            for line in process.stdout:
                log.write(line)
                if echo:
                    print(line, end='', flush=True)
            rc = process.wait()
        except BaseException:
            try:
                os.killpg(process.pid, signal.SIGTERM)
                process.wait(timeout=10)
            except ProcessLookupError:
                pass
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            raise
        finally:
            process.stdout.close()
        if rc:
            raise RuntimeError(f'処理が終了コード{rc}で停止しました。ログ: {path}')


def completed_training(workspace, run_id):
    root = Path(workspace)
    report_path = root/'experiments'/run_id/'training_summary.json'
    adapter = root/'adapters'/(run_id+'_stage2')/'adapter_model.safetensors'
    if not report_path.exists():
        return False
    report = json.loads(report_path.read_text())
    if report.get('complete') is not True or not adapter.is_file():
        raise RuntimeError('完了記録とStage 2の保存状態が一致しません。')
    if sha256_file(adapter) != report['stage2_weights_sha256']:
        raise RuntimeError('完了時からStage 2の重みが変更されています。')
    return True

WORKSPACE = '/content/drive/MyDrive/geniac_workspace'
RUN_ID = 't4_20261013_run02_prmfull'
if not Path(WORKSPACE).is_dir():
    raise RuntimeError('元のランタイムのDrive接続を確認してください。')
if not completed_training(WORKSPACE, RUN_ID):
    log = Path(WORKSPACE)/'experiments'/RUN_ID/'logs/train.log'
    run_logged([sys.executable, '-u', str(worker), 'train'], log)
if not completed_training(WORKSPACE, RUN_ID):
    raise RuntimeError('学習完了を確認できませんでした。')
print('RUN02_TRAINING_COMPLETE')
print(Path(WORKSPACE)/'experiments'/RUN_ID/'training_summary.json')
